# Geospatial Data Prep

**Goal**: 
1. Build `dim_zipcode_geo` using the `uszipcode` library.
2. Enrich `slv_pharmacy_network` with Lat/Long.

In [14]:
import sqlite3
con = sqlite3.connect("simple_db.sqlite")

In [15]:
cur = con.cursor()

In [26]:
cur.execute("SELECT * FROM simple_zipcode where state = 'FL'")
res = cur.fetchall()

In [1]:
# import sqlalchemy_mate as sam
# from sqlalchemy_mate.api import ExtendedBase  # vị trí mới
# sam.ExtendedBase = ExtendedBase


from uszipcode import SearchEngine
import pandas as pd
from pyspark.sql import functions as F, types as T

# Initialize SearchEngine
search = SearchEngine(
    # simple_or_comprehensive=SearchEngine.SimpleOrComprehensiveArgEnum.comprehensive
)

c:\Users\nghia.n\AppData\Local\anaconda3\envs\spark-env\lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')
c:\Users\nghia.n\AppData\Local\anaconda3\envs\spark-env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [12]:
list_states = [ "OH", "NE", "IL", "MO", "KS", "NM", "AZ", "CO", "TX", "HI", "IA", "WA", "NY", "MS", "AL", "MT", "GA", "MA", "MI", "TN", "AR", "KY", "WI", "CA", "CT", "NV", "VA", "FL", "IN", "DE", "PA", "ME", "NH", "NJ", "DC", "MD", "NC", "RI", "OR", "MN", "UT", "OK", "SC", "LA", "SD", "ID", "WV", "ND", "WY", "VT", "PR"]


res = []

for st in list_states:
    res += search.by_state(st, returns=None)

In [14]:
# 1. Extract Valid US Zipcodes
# We will iterate states or just dump all if possible. 
# uszipcode stores data in a local sqlite. We can query it.


data = []
for r in res:
    data.append({
        "zip_code": r.zipcode,
        "city": r.major_city,
        "state": r.state,
        "county": r.county,
        "lat": r.lat,
        "lng": r.lng,
        "population": r.population,
        "density": r.population_density
    })

pdf_zips = pd.DataFrame(data)

# df_zips = spark.createDataFrame(pdf_zips)
# df_zips = df_zips.withColumn("zip_code", F.lpad(F.col("zip_code"), 5, "0"))

# # Save as Dimension
# df_zips.write.format("delta").mode("overwrite").saveAsTable("cms_partd_gold.mart.dim_zipcode_geo")
# display(df_zips)

In [18]:
pdf_zips

,zip_code,city,state,county,lat,lng,population,density
0,43001,Alexandria,OH,Licking County,40.08,-82.61,2400.0,90.0
1,43003,Ashley,OH,Delaware County,40.40,-82.95,2917.0,86.0
2,43004,Blacklick,OH,Franklin County,40.02,-82.80,22727.0,1757.0
3,43006,Brinkhaven,OH,Holmes County,40.47,-82.19,822.0,34.0
4,43009,Cable,OH,Champaign County,40.17,-83.64,2135.0,62.0
...,...,...,...,...,...,...,...,...
29925,00979,Carolina,PR,Carolina Municipio,18.44,-66.03,NaN,NaN
29926,00982,Carolina,PR,Carolina Municipio,18.41,-65.99,NaN,NaN
29927,00983,Carolina,PR,Carolina Municipio,18.40,-65.98,NaN,NaN
29928,00985,Carolina,PR,Carolina Municipio,18.41,-65.95,NaN,NaN


In [16]:
pdf_zips.to_csv("dim_zipcode_geo.csv", index=False)

In [ ]:
# 2. Calculate Distance Function (UDF or Spark Native)
# Haversine formula

def haversine_expression(lat1, lon1, lat2, lon2):
    # Returns distance in miles
    R = 3958.8 # Radius of Earth in miles
    
    dlat = F.radians(lat2 - lat1)
    dlon = F.radians(lon2 - lon1)
    a = F.sin(dlat / 2)**2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2)**2
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
    return R * c